## Data Preprocessing

In the previous session, you successfully acquired and prepared a subset of the ESS10 data 

Keep in minde we aim to the following hypothesis:

> **Centrist individuals exhibit lower levels of affective polarization.**

Today, you will clean this dataset to make it analysis-ready. This involves:

1. Loading the subset data you created last time
2. Filtering observations (focusing on France)
3. Recoding variables for analysis
4. Creating relevant variables for the analysis
5. Saving the cleaned dataset

### Tips

- You should adapt code from previous [notebooks](https://github.com/mickaeltemporao/materials/tree/main), such as:
    - `05-data-exploration-rows.ipynb`
    - `06-data-management-existing-values.ipynb`


## Loading the Data

We have prepared a helper function in the `code/preprocess.py` module that will load your subset data. If the subset file doesn't exist, it will recreate it from the raw ESS data.

In [ ]:
# Import necessary libraries
import sys
import pandas as pd

# Adding the code directory to path
sys.path.append('../code')  

# Import the preprocessing module
from preprocess import subset

In [ ]:
# Load the subset data using our helper function
df = subset()
df.sample(5)

In [ ]:
# Quick sanity check before we get started
df.describe()

# Let's Start!
## Filtering French Observations

Since some variables are country-specific, let's filter our dataset to focus on French respondents.

**Task:**
1. Filter the dataset to include only respondents from France (`country == 'FR'`)
2. Create a new dataframe called `df_france`
3. Drop the `cntry` column
4. Display the first few rows

In [ ]:
# Your code here
mask = df['cntry'] == "FR"
df_france = df[mask]

df_france = df_france.drop(columns='cntry')
# SAME AS 
# df_france = df_france.drop('cntry', axis = 1)

df_france

## Filtering Relevant Observations

Before we can analyze the data, we need to understand how the variables are coded. 

If you check the codebook, we see that most variables have values that are not applicable to our analysis (66, 77, 88, ...). 
For now, we will simply remove irrelevant observations.

> The codebook has been automatically downloaded in `data/raw/ESS10 codebook.html`

**Task:**
1. Filter each variable to include only relevant observations
2. Check the values of the remaining observations
2. Display the first few rows of the dataset

In [ ]:
# Your code here (you can create multiple code blocks)
df_france.describe()

In [ ]:
# quick & dirty cleaning ...
mask = df_france > 12
df_france[mask] = pd.NA
df_france

# Also possible using newer pandas methods
# df_france.mask(df_france > 12, pd.NA)

In [ ]:
df_france.dropna()

In [ ]:
df_france = df_france.dropna()

## Creating the Centrist Variable 

For our hypothesis, we need to identify "centrist" individuals. 
Therefore, we will should add a new variable capturing this concept into our data frame.

**Task:**
- Use the `lrscale` to create a centrism variable
- Add the new variable `centrism` to the data frame
- Check the distribution of the newly created variable


In [ ]:
# Your code here:
# Create the centrism variable
df_france['lrscale'].value_counts().sort_index().plot.bar()

df_france['centrism'] = df_france['lrscale'].between(4,6).astype(int)

# Check the distribution
df_france['centrism'].value_counts().plot.bar()


In [ ]:
# Cross-tab with ideology to verify
pd.crosstab(df_france['lrscale'], df_france['centrism'])

## Creating an Affective Scale?

For our hypothesis, we also need to build an affective evaluation scale.

Unfortunately, ... the ESS does not have direct out-party dislike or feeling thermometers toward parties.
Not having direct observations of what we are trying to test, is common when doing research. 

We need need to find a way around this by building a proxy that is close to our original concept.
One way to build such proxy is by creating an additive scale combining multiple variables of interest. 

That is, we could add variables together to build an "Affective Scale".

**Task:**
- Select some or all trust and satisfaction variables.
- Make sure they are coded in the same direction (higher = more positive evaluation).
- Combine them into an additive scale (sum or average).
- Create a new variable called `aff_eval` in the data frame.
- Check the distribution of aff_eval using summary statistics and a histogram.


In [ ]:
# Your code here:
# Create the aff_eval variable
aff_vars = ["trstplt", "trstprt", "stfgov", "stfdem"]
df_france[aff_vars].describe()
df_france[aff_vars].sum(axis=1)
df_france[aff_vars].sum(axis=1)
df_france[aff_vars].sum(axis=1) / 40 
df_france[aff_vars].sum(axis=1).describe()
df_france["aff_eval"] = df_france[aff_vars].sum(axis=1) / 40

# Check the distribution
df_france["aff_eval"].plot.hist()


## Exploratory Data Analysis

Now let's explore our cleaned data to understand the relationships between variables.

In [ ]:
# Summary statistics for key variables
main_vars = ['aff_eval', 'centrism']
df_france[main_vars].describe()


In [ ]:
# Explore the relationships between the newly created variables
df_france.groupby('centrism')['aff_eval'].mean()


In [ ]:
df_france.groupby('centrism')['aff_eval'].plot.hist()


## Saving the Final Dataset

Let's create our final analysis-ready dataset with the key variables for testing our hypothesis.

In [ ]:
# Save the final cleaned dataset
df.to_csv("data/ess_clean.csv", index=False)
